# 🫀 퀘스트 46 · Q4-M — **P-on-T · 환자 내 군집 · 잔차 부스팅**

| | **MedKOS / `notebooks/quest46_q4m_pont_cluster_boost.ipynb`** |
|---|---|
| 퀘스트 | `ailab-2026-0046` — 작동하는 축(형태) **위에** 얹는다 |
| 부모 런 | `quest46_q4l_delineation_audit`(`20260805T1208`) |
| 예상 소요 | **35–45분**(CPU) + 잔차 부스팅 **10–15분**(GPU 있을 때만) |

## ★★★ Q4-L — 자를 고치니 **부호가 뒤집혔다**(다만 작다)

```
팔        차원  k-스윕   달성률@300  AUROC   PR-AUC
base       9   0.6791   0.8074    0.9420  0.5796
morph     17   0.8361   0.9018    0.9529  0.7236   ← 유일하게 크다
morphp    23   0.8391   0.8882    0.9508  0.7300   ← 형태 확장은 거의 0
intv2     21   0.6882   0.8146    0.9360  0.5697
ishuf2    21   0.6685   0.8006    0.9379  0.5573
full2     29   0.8202   0.8875    0.9503  0.7038

intv2 − ishuf2  k-스윕 **+0.0197**   (Q4-K −0.0163 → **부호 반전**)
intv2 − base    k-스윕 +0.0091       (차원 비용 ishuf2−base −0.0106 을 겨우 상쇄)
full2 − morph   k-스윕 **−0.0159**   ← 구간을 형태 위에 얹으면 **나빠진다**
morphp − morph  k-스윕 +0.0030 · 달성률@300 **−0.0136**  ← 형태도 포화
```

자 검증은 전부 통과했다 — QRS **77.8ms**(44.4) · PR **147.2ms**(104.2) · **역위 P 14.9%** ·
P|corr| N 0.924 > V 0.834 · 리드 분리 **62.8%**. 그리고 **회고 검증에서 `tp_over_rr` 의
레코드 내 |ρ| 가 1.0000** 으로 잡혀 새 중복 감사가 작동함이 확인됐다(풀링 R² 는 0.7747).

### 블록별 — **하나만 살아남았다**

```
(a) P-P/ΔPR    +0.0029 [−0.0110, +0.0171]
(b) P폭/QRS폭   −0.0110 [−0.0369, +0.0117]   ← 처음 제대로 시험됨
(c) P폭/T폭     +0.0015 [−0.0011, +0.0044]
(d) TP 잔차     −0.0070 [−0.0168, +0.0017]
(e) P 형태      +0.0054 [−0.0308, +0.0413]
(f) **역위 P·검출 +0.0082 [+0.0027, +0.0151]**  ← **유일하게 CI 하한 > 0**
```

**(f) 는 내가 버그를 고치며 새로 생긴 열**(`p_polarity`·`p_found`)이다. 작지만 유일하게
0 을 벗어났다 — **P 의 극성**이 살아남은 유일한 구간 신호다.

## ★★★ 형태가 **정확히 무엇을 봤나**

```
단변량 |AUROC−0.5|   p_energy_ratio 0.2361 > corr_st_min 0.2067 > corr_full_min 0.2053
                     > amp_ratio 0.1764 > area_ratio 0.1652 > corr_qrs 0.146/0.143
                     > qrs_width 0.1150(최하)
위양성 구성          심실기원 4380 → **1444**(67%↓) · 상심실정상 6338 → 8166(빈자리를 채움)
k 별 이득            k=50 **+0.2230** · k=100 +0.1810 · k=200 +0.1300 · k=300 +0.0950
형태 단독(`monly`)   k-스윕 0.6199 — **형태만으론 S 를 못 찾는다**
```

**형태는 「S 를 더 찾는 것」이 아니라 「순위 맨 위에서 V 를 밀어내는 것」**이다. 그리고
그 안에서 가장 강한 건 **선행 P 창 에너지**(V 는 방실해리라 선행 P 가 없다)와 **ST-T 상관**
(V 의 2차성 재분극 변화)이다 — **QRS 폭이 최하위**인 게 오히려 그 해석을 뒷받침한다.
⚠️ `full − fshuf` +0.0932 는 두 번째 결과가 아니다 — `full − morph` 가 −0.0159 이므로
**그 +0.0932 의 사실상 전부가 형태 몫**이다.

## ★★★ 왜 딥러닝이 CPU 기준선보다 나빴나 — **딥러닝 탓이 아니다**

```
cpu_base 0.8119 · cpu_full 0.8810 · dl_wave 0.4940 · dl_hybrid 0.8287 (−0.0523)
```

### ① 하이브리드의 **손수특징 부분이 sklearn 과 다른(더 나쁜) 회귀**였다

같은 특징·같은 데이터로 절차만 바꿔 재보면:

```
sklearn lbfgs(완전수렴 · L2 C=1.0 · 클래스가중 없음)       AUROC 0.8434  ← cpu_full 절차
Adam 4에폭 · 정규화 없음 · pos_weight 6.1                AUROC 0.7008  ← Q4-K 하이브리드 절차
Adam 4에폭 · weight_decay 1e-4 · pos_weight 제거          AUROC 0.7199
Adam 40에폭 · weight_decay 1e-4 · pos_weight 제거         AUROC 0.8433  ← 회복
```

**−0.14 가 순전히 최적화 절차에서 나왔다.** 4에폭 미수렴 · 무정규화 · `pos_weight`
(순위가 아니라 균형 로그손실을 최적화)의 3중 실수다.

### ② CNN 입력에 **조기성이 아예 없다**

입력이 `beat (2, 300)` 인데 **R 피크가 항상 index 100 에 정렬**돼 있다. 즉 **RR 간격이
입력에 존재하지 않는다.** SVEB 의 정의가 조기성인데 **정의상 볼 수 없는 입력**을 줬다.
`dl_wave` 0.4940 이 그 결과다.

### ③ 유효 표본이 **레코드 56개**다

박동 18만은 착시다 — LORO/5겹은 **처음 보는 레코드**에 시험한다. 딥러닝이 손수 특징을
이기려면 보통 **수천 명**이 필요하다(문헌 1D CNN 74.56% 도 MIT-BIH 44명).

⇒ **선생님 직관이 맞다.** 「기준을 잡고 그 위에 미묘한 것을 얹는」 구조를 안 만든 게
문제였다. 이번엔 **잔차 부스팅**으로 짓는다:

```
logit = logit_cpu(**고정 offset**) + α · CNN(다중 박동 + RR 채널)      α 초기값 0
```

α=0 에서 시작하므로 **최소한 `cpu_full` 만큼은 보장**되고, 그 위에서만 개선한다.

## 이번에 넣는 새 축 둘

### ★★★ 축 A — **P-on-T**(직전 박동의 T 창)

매우 이른 PAC 은 **P′ 가 선행 T 파에 겹친다**. 그러면 **직전 박동의 T 파 형태가 변형**된다.
그런데 지금까지 모든 특징은 **그 박동 자신의 창**만 봤다 — **직전 박동의 T 창을 이 박동의
특징으로 넣은 적이 없다.** 유일하게 작동하는 축(형태)의 **시간적 확장**이고, 임상 근거가
명확하다(P-on-T · blocked PAC 의 단서도 여기다).

### ★★ 축 B — **환자 내 이소성 군집**(무라벨 개인화)

한 환자의 S 는 대개 **소수의 초점**에서 나오므로 **서로 닮았다**. 레코드 안에서
(형태+구간) 좌표를 **라벨 없이 군집화**하면 「다수 군집(정상)」과 「소수 이탈 군집」이
나온다. 그 이탈 정도를 특징으로 쓴다 — **「이 환자의 PAC 은 다 비슷하게 생겼다」** 는
실제 판독 방식이다.

## 관문 (사전등록)

| 관문 | 무엇 | 통과 기준 |
|---|---|---|
| **Q0** | 자 검증(Q4-L + **P 폭·검출 강화**) | 깨지면 **중단** |
| **Q1** | 중복 감사 v2(레코드 내 순위) | 입구 검사 |
| **Q2 ★★★ 주 관문** | `pont` vs `pshuf` — **형태 위에** P-on-T 6열 | 측정된 raw 영점 상단 초과 |
| **Q3 ★★** | `clus` vs `cshuf` — **형태 위에** 군집 4열 | 배포·기전 병기 |
| **Q4 ★★** | `comb` vs `morph` — 둘 다 얹으면 현재 최선을 넘는가 | 배포·기전 병기 |
| **Q5 ★★** | **잔차 부스팅**(GPU 있을 때만) — α=0 출발이라 하한이 보장된다 | 관문 아님 |
| **Q6** | 운영점·적응증 · 검산표 | R38 ⑦ · R41 ② |

⚠️ **새 데이터 0** · **리듬 라벨은 없다**(Q7-D 확정 — 원본 주석이 비어 있다. AF 는 RMSSD 대리뿐).


In [ ]:
# CELL 0 — 공용 사전점검
import numpy as np

def decide(lo, hi, thr, direction):
    if direction not in (">", "<"):
        raise ValueError("direction 은 '>' 또는 '<'")
    if not (np.isfinite(lo) and np.isfinite(hi) and np.isfinite(thr)):
        return "⚠️ 미결"
    if direction == ">":
        if lo > thr: return "✅ 지지"
        if hi < thr: return "❌ 기각"
    else:
        if hi < thr: return "✅ 지지"
        if lo > thr: return "❌ 기각"
    return "⚠️ 미결"

def mde(lo, hi):
    return (hi - lo) / 2.0 if np.isfinite(lo) and np.isfinite(hi) else float("nan")

def boot_mean(v, seed, nb=3000, q=2.5):
    d = np.asarray(v, float); d = d[np.isfinite(d)]
    if len(d) < 3:
        return float("nan"), float("nan"), float("nan"), len(d)
    rng = np.random.RandomState(seed)
    b = [d[rng.randint(0, len(d), len(d))].mean() for _ in range(nb)]
    return (float(d.mean()), float(np.percentile(b, q)),
            float(np.percentile(b, 100 - q)), len(d))

def boot_pair(a, b, seed, nb=3000, q=2.5):
    a = np.asarray(a, float); b = np.asarray(b, float)
    m = np.isfinite(a) & np.isfinite(b); a, b = a[m], b[m]
    if len(a) < 3:
        return float("nan"), float("nan"), float("nan"), len(a)
    rng = np.random.RandomState(seed)
    d = [(b[j] - a[j]).mean() for j in (rng.randint(0, len(a), len(a)) for _ in range(nb))]
    return (float((b - a).mean()), float(np.percentile(d, q)),
            float(np.percentile(d, 100 - q)), len(a))

def need_super(n, half, eff, p80=False):
    if not np.isfinite(half) or not np.isfinite(eff) or abs(eff) < 1e-9 or n < 1:
        return float("nan")
    r = float(n) * (half / abs(eff)) ** 2
    return r * 2.04 if p80 else r

class AssetError(RuntimeError): pass
print("CELL 0 ✅")


In [ ]:
# CELL 1 — 설정 · 사전등록
import os, sys, json, importlib, time, warnings
importlib.invalidate_caches(); warnings.filterwarnings("ignore")

SMOKE = os.environ.get("MEDKOS_SMOKE") == "1"
_ENV_ROOT = os.environ.get("MEDKOS_DRIVE_ROOT")
if _ENV_ROOT:
    DRIVE_ROOT = _ENV_ROOT
else:
    try:
        from google.colab import drive; drive.mount("/content/drive", force_remount=False)
        DRIVE_ROOT = "/content/drive/MyDrive"
    except Exception as e:
        print("Colab 아님:", e); DRIVE_ROOT = "/content"
PROJECT = os.path.join(DRIVE_ROOT, "MedKOS", "ecg-model")
MITBIH  = os.path.join(DRIVE_ROOT, "mitbih")
sys.path.insert(0, os.path.join(PROJECT, "lib"))
from medkos_run import MedKOSRun

SEED0, IDX_S = 20260805, 1
RHY_K = (5, 10, 20, 32)
MIN_S, MIN_N = 25, 25
DEV_EVERY = 4
K_SWEEP = (50, 100, 200, 300)        # ★★★ 주 지표 = 이 네 점의 평균 달성률
MAIN_K = 300
MAX_NEG_SLOPE, MIN_AUC_SLOPE = 0.10, 0.55
FS = 360.0
R_IDX = 100
W_P_S  = (20, 88)
W_T_S  = (135, 265)
W_Q_S  = (72, 148)
FRAC_QRS, FRAC_P, FRAC_T = 0.10, 0.25, 0.25   # ★ P·T 는 봉우리 확장으로 되돌렸다
LAG = 30
P_CORR_MIN = 0.35   # 하한
P_GAP = 4           # ★ P 창 상단을 **q_on 보다 이만큼 앞**으로 자른다(QRS 를 물지 않게)
P_NULL_Q = 0.95     # ★ 레코드별 **영점 교정** 문턱의 백분위(시간 역전 템플릿)
TMPL_LO, TMPL_HI, TMPL_MIN = 0.92, 1.08, 30
DUP_RHO = 0.85                       # ★ 중복 감사 v2 — 레코드 내 순위 상관 문턱
QRS_LO_MS, QRS_HI_MS = 70.0, 130.0   # ★ 자 검증 범위(사전 고정)
PR_LO_MS, PR_HI_MS = 110.0, 230.0
NB_BOOT = 400 if SMOKE else 2000
N_PERM  = 2   if SMOKE else 6

ARMS = ("base", "morph", "pont", "pshuf", "clus", "cshuf", "comb")
MAIN_CT = "pont-pshuf"
CONTRASTS = (("morph-base",  "base",   "morph"),
             ("pont-pshuf",  "pshuf",  "pont"),
             ("pont-morph",  "morph",  "pont"),
             ("clus-cshuf",  "cshuf",  "clus"),
             ("clus-morph",  "morph",  "clus"),
             ("comb-morph",  "morph",  "comb"))
N_CLUS = 3                      # ★ 레코드 내 무라벨 군집 수(사전 고정)
PW_LO_MS, PW_HI_MS = 60.0, 130.0    # ★ Q4-L 이 138.9ms 로 넓게 나왔다 — 범위를 건다
# ★ 검출률 상한 대신 **V 에서 검출률이 낮아야 한다**를 건다(더 의미 있는 검사)
DL_FOLDS, DL_EPOCH, DL_BATCH, DL_EMB, DL_WD = 5, 12, 1024, 16, 1e-4
PRIMARY, SECONDARY = "ksw", "auc"
READ_ORDER = ("P0", "P1", "P2", "P3", "P4", "P5", "P6")
SV5 = os.path.join(MITBIH, "svdb_data5.npz")

REF = dict(
    q4k=dict(base=dict(ach=0.8074, auc=0.9420, ap=0.5796),
             morph=dict(ach=0.9018, auc=0.9529, ap=0.7236),
             intv=dict(ach=0.7810, auc=0.9319, ap=0.5442),
             ishuf=dict(ach=0.7973, auc=0.9389, ap=0.5591),
             full=dict(ach=0.8856, auc=0.9509, ap=0.7007),
             gate=(-0.0163, -0.0465, 0.0096), thr=0.0520,
             blocks={"(a)": 0.0002, "(b)": -0.0050, "(c)": -0.0185,
                     "(d)": -0.0026, "(e)": -0.0285},
             pr_ms=104.2, qrs_ms=44.4, pw_ms=75.0, p_found=0.984,
             tp_r2=0.020, tp_uni=0.4269, base_uni=0.4337, pfound_uni=0.0182,
             fp_v=dict(base=4380, morph=1444, intv=4117, full=1479),
             dl=dict(wave=0.4940, hybrid=0.8287, cpu_full=0.8810, gain=-0.0523)),
    lit=[("de Chazal 2004", 0.759, 0.385), ("Llamedo 2011", 0.77, 0.39),
         ("1D CNN inter-patient", 0.7456, None)])

RULE_CHECK = {
    "R11 매크로":       "환자 단위 · 상한과 함께 읽는다",
    "R16 fallback 없음": "**자 검증(P0)이 깨지면 중단**한다 — 고장난 자로 잰 결과는 무효다",
    "R22 누출 없음":     "LORO · 템플릿·델리네이션·잔차화는 라벨을 안 쓴다. `sym` 은 **자 검증에만**",
    "R26 영점":         "영점은 raw(비교정) · rep 수준 산포 병기 · 흐리면 기각을 미결로 강등",
    "R29 ② 분기 금지":   "P0 이 깨지면 아래를 **안 읽는다**",
    "R33 ① MDE":        "관문마다 MDE. 미결 ≠ 등가",
    "R34 ② 문턱 금지":  "창·임계·k-스윕·중복문턱을 **사전 고정**. 지표 변경도 데이터 보기 전",
    "R35 ① 자 먼저":    "★★★ **이 런 전체가 자를 고치는 런**이다 — Q4-K 는 가설이 아니라 자가 틀렸다",
    "R34 ② 창 자동화":  "★★★ **P 창 상단을 q_on 에서 자동 유도**한다 — 고정 창이 QRS 를 물면 "
                        "봉우리가 QRS 상승부가 된다(1차 실행이 그래서 죽었다)",
    "R36 ⑤ 자기검사":   "★★ **Q4-L 의 과잉 수정을 되돌린다** — 첫~마지막 교차는 다상(QRS)에만 "
                        "옳고 단상(P·T)은 봉우리 확장이 맞다(Q4-L P 폭 138.9ms 는 창 전체를 삼킨 것)",
    "R38 ⑦ 요약 정합":  "★★★ 버그 넷을 **명시**한다 — 구간 축은 기각이 아니라 **미시험**이었다",
    "R39 ① 대안설명":   "고쳐도 안 되면 그때가 기각이다 — 그 경우 남는 축은 **형태·적응증**",
    "R41 ② 0 근처":     "효과가 0 근처면 필요표본은 해석 불가",
}

CONFIG = dict(
    exp="quest46_q4m_pont_cluster_boost", quest="ailab-2026-0046", step="delineation-audit",
    parent_exp=["quest46_q4k_pp_interval"],
    purpose=("★★★ **Q4-K 의 구간 축은 기각이 아니라 미시험이었다 — 자가 넷이나 고장나 있었다.** "
             "① **역위 P 를 구조적으로 못 찾는다**: P 정렬을 `max(부호 있는 상관)` 으로 했는데 "
             "**하부 심방·접합부 초점의 이소성 P 는 역위**라 정상 P 템플릿과 상관이 **−0.98** 이고 "
             "잡음(+0.2)에 **진다**. **우리가 찾으려는 바로 그 박동에서 검출이 실패**한다"
             "(`p_found` 단변량 0.0182 무정보 · V 위양성 4380 → 4117 로 6% 감소, 형태는 67%). "
             "⇒ `|corr|` 로 정렬하고 **`p_polarity` 를 특징으로** 낸다 — 역위 P 는 버그가 아니라 "
             "**소견**이다. ② **QRS 폭이 실은 R 파 폭**이었다: `|x−등전위|` 의 봉우리 연속 확장은 "
             "Q–R–S **3상**의 0 교차에서 끊긴다(합성: 실제 78ms → 36ms · Q4-K 실측 44.4ms). "
             "⇒ **첫~마지막 임계 교차**로 고친다. **선생님 가설 (b) P폭/QRS폭 은 시험된 적이 없다.** "
             "③ **중복 감사가 풀링·선형**이었다: `tp_over_rr` 는 풀링 R² 0.020(=새 축)인데 단변량 "
             "0.4269(기존 RR 최고 0.4337 급) — **둘이 같이 큰 게 단조 중복의 지문**이다. 레코드마다 "
             "템플릿 기하가 달라 풀링 R² 가 레코드 간 변동에 지배되는데, 매크로 채점은 **레코드 내 "
             "순위**만 쓴다(합성 재현: 풀링 R² 0.357 vs 레코드 내 스피어만 0.932). ⇒ **레코드 내 "
             "순위 상관**으로 바꾼다. ④ **리드를 QRS 진폭으로 골랐다** — P 는 다른 리드에서 보인다. "
             "PR 도 「P 봉우리 → R 봉우리」였다(표준은 P 시작 → QRS 시작 · 실측 104.2ms 로 생리 범위 "
             "밖). ★★★ **그리고 관문 자체가 통과 불가에 가까웠다**: morph 달성률 0.9018 이라 남은 "
             "여지가 0.0982 인데 문턱이 **+0.0520**(여지의 53%)이었다. ⇒ 주 지표를 **k-스윕 평균 "
             "달성률**(k ∈ {50,100,200,300})로 바꾼다 — 항등식 `달성률 = TP/min(S,k)` 이라 k ≤ S 에선 "
             "**정밀도@k**, k > S 에선 재현율이고, 네 점 평균이라 **영점 분산도 준다**. "
             "★ **딥러닝은 접는다** — Q4-K 에서 `dl_wave` 0.4940 · `dl_hybrid − cpu_full` "
             "**−0.0523** 로 문헌대로였다."),
    dataset="SVDB — svdb_data5.npz (새 데이터 0 · GPU 0)",
    arms=list(ARMS), main_contrast=MAIN_CT, primary=PRIMARY, secondary=SECONDARY,
    k_sweep=list(K_SWEEP), fs=FS, r_idx=R_IDX,
    windows=dict(p=W_P_S, t=W_T_S, q=W_Q_S, frac=(FRAC_QRS, FRAC_P, FRAC_T), lag=LAG,
                 p_corr_min=P_CORR_MIN),
    guards=dict(dup_rho=DUP_RHO, qrs=(QRS_LO_MS, QRS_HI_MS), pr=(PR_LO_MS, PR_HI_MS)),
    read_order=READ_ORDER, dev_every=DEV_EVERY, n_boot=NB_BOOT, n_perm=N_PERM,
    smoke=SMOKE, ref=REF, rule_check=RULE_CHECK,
    predictions={
        "P0": "★★★ **자 검증(중단 관문)** — QRS 폭 70~130ms · PR 110~230ms · **역위 P 를 "
              "실제로 잡는가** · **P 검출 타당성**(V 박동에서 |corr| 이 N 보다 낮아야 한다). "
              "하나라도 깨지면 **중단**하고 구간 축을 접는다",
        "P1": "★★★ **중복 감사 v2** — **레코드 내 스피어만 |ρ|**(단조·순위 불변). Q4-K 의 12열을 "
              "**회고 재판정**해서 `tp_over_rr` 가 ⛔ 로 잡히는지 직접 확인한다",
        "P2": "★★★ **주 관문** — `pont` vs `pshuf`(차원 동일) · **형태 위에** P-on-T 6열",
        "P3": "★★ `clus` vs `cshuf` · **`comb` vs `morph`** — 현재 최선 위에 얹히는가",
        "P4": "★★ 선생님 가설 (a)~(f) 블록별 — **(b) P폭/QRS폭 은 처음 제대로** 시험된다",
        "P5": "★★ **잔차 부스팅**(GPU 있을 때만) — `logit = logit_cpu(고정) + α·CNN(다중박동+RR)`. "
              "α=0 출발이라 **최소한 cpu 만큼은 보장**된다",
        "P6": "필요표본 · 검산표"},
    caveat=("★★★ **이 런은 새 가설을 얹지 않는다 — 고장난 자를 고치고 검증한다**(R35 ①). "
            "Q4-K 의 자기검증 수치(PR 104.2ms · QRS 44.4ms)가 이미 고장을 말하고 있었는데 "
            "guard 가 60~260ms 로 너무 느슨해 통과시켰다. 이번엔 **생리 범위를 좁게** 잡고 "
            "**역위 P 검출**과 **P 검출 타당성**까지 중단 관문에 넣는다. "
            "★★ **`sym` 을 자 검증에 쓴다** — V 박동에서 P 상관이 낮아야 한다는 확인이다. "
            "특징 생성·모델 적합·열 선택에는 **일절 안 쓴다**(R22). Q7-P0 의 생리학적 "
            "자기검증과 같은 용법이다. "
            "★★ **고쳐도 안 되면 그때가 진짜 기각**이다(R39 ①) — 그 경우 남는 축은 "
            "**형태(+0.0944 · 두 런 재현)** 와 **적응증 좁히기(달성률 0.9533)** 다. "
            "★ **딥러닝은 접는다** — Q4-K 에서 `dl_hybrid − cpu_full` −0.0523 이었다."))
np.random.seed(SEED0)
run = MedKOSRun("quest46_q4m_pont_cluster_boost", CONFIG, project=PROJECT)
run.log("설정 ✅ **Q4-L — 자를 고치고 검증한다: 델리네이션 감사**")
run.log(f"  ★★★ Q4-K 는 **가설이 기각된 게 아니라 자가 넷이나 고장나 있었다**")
run.log(f"     ① 역위 P 미검출(이소성의 대표 소견인데 `max(부호 상관)` 이라 잡음에 진다)")
run.log(f"     ② QRS 폭이 실은 **R 파 폭**(3상 0교차에서 확장이 끊긴다 · 실측 {REF['q4k']['qrs_ms']}ms)")
run.log(f"     ③ 중복 감사가 **풀링·선형**(채점은 레코드 내 순위로 한다)")
run.log(f"     ④ 리드를 QRS 진폭으로 골랐고 PR 이 「봉우리→봉우리」였다(실측 {REF['q4k']['pr_ms']}ms)")
run.log(f"  ★★★ 그리고 **관문이 통과 불가에 가까웠다** — 남은 여지 0.0982 인데 문턱 "
        f"+{REF['q4k']['thr']:.4f}(53%)")
run.log(f"  ⇒ 주 지표를 **k-스윕 평균 달성률** k ∈ {list(K_SWEEP)} 로 바꾼다 "
        f"(달성률 = TP/min(S,k) — k ≤ S 면 정밀도@k)")
if SMOKE:
    run.log(f"  ⚠️ **스모크런** — 비용 손잡이만 축소(NB_BOOT={NB_BOOT} · N_PERM={N_PERM})")
run.log("\n  사전등록 규칙 체크리스트 (R29 ③)")
for k_, v_ in RULE_CHECK.items():
    run.log(f"    [x] {k_:<18} {v_}")


In [ ]:
# CELL 2 — 【P-0】 코호트 · ★★★ 고친 델리네이션 · 특징
import pandas as pd
from collections import Counter
from scipy.stats import spearmanr
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.metrics import average_precision_score, roc_auc_score

run.log("\n" + "=" * 100)
run.log("【P-0】 ★★★ 고친 델리네이션 — 역위 P · QRS 3상 · 리드 분리")
run.log("=" * 100)
VERD, NOTE = {}, {}
def g_(k, v, d):
    VERD[k] = v; NOTE[k] = d; run.log(f"  {k:<5}{v}  {d}")

if not os.path.exists(SV5):
    raise AssetError(f"{SV5} 없음(R16)")
D5 = np.load(SV5, allow_pickle=True)
for need in ("pid", "y3", "pre_rr", "post_rr", "beat", "sym"):
    if need not in D5.files:
        raise AssetError(f"`{need}` 가 자산에 없다(R16)")
PID = np.asarray(D5["pid"]).astype(int); Y3 = np.asarray(D5["y3"]).astype(int)
PRE = np.asarray(D5["pre_rr"], float); POST = np.asarray(D5["post_rr"], float)
K = np.where(Y3 >= 0)[0]
RID = PID[K]; TT_ = (Y3[K] == IDX_S)
pre = PRE[K].astype(float); post = POST[K].astype(float)
BEAT = np.asarray(np.asarray(D5["beat"])[K], dtype="float32")
SYM = np.asarray(D5["sym"]).astype("<U2")[K]
NL, LW = BEAT.shape[1], BEAT.shape[2]
if LW <= W_T_S[1]:
    raise AssetError(f"파형 길이 {LW} 가 T 창 {W_T_S} 보다 짧다(R34 ②)")
RS = np.array(sorted(set(RID.tolist())))
IDX_ALL = {int(r): np.where(RID == r)[0] for r in RS}
run.log(f"  파형 {BEAT.shape} · 리드 {NL}")

_S = pd.Series(pre); _G = _S.groupby(pd.Series(RID))
def local_base(k):
    r = np.asarray(_G.apply(lambda x: x.shift(1).rolling(k, min_periods=1).median())).astype(float)
    return np.where(np.isfinite(r), r, pre)
_med = _G.transform("median").to_numpy()
_std = _G.transform("std").to_numpy(); _mean = _G.transform("mean").to_numpy()
BASE12 = local_base(12); REL = pre / (BASE12 + 1e-9)
F_BASE = np.nan_to_num(np.c_[_med - pre,
                             np.column_stack([1.0 - pre / (local_base(k) + 1e-9) for k in RHY_K]),
                             post - pre, np.nan_to_num(_std / (_mean + 1e-9)),
                             np.log1p(np.clip(pre, 0, None)), np.log1p(np.clip(post, 0, None))],
                       nan=0.0, posinf=0.0, neginf=0.0)

def corr_to(x, t):
    xc = x - x.mean(-1, keepdims=True); tc = t - t.mean(-1, keepdims=True)
    return (xc * tc).sum(-1) / (np.sqrt((xc ** 2).sum(-1) * (tc ** 2).sum(-1)) + 1e-9)

def peak_of(x, lo, hi, base):
    seg = np.abs(x[lo:hi] - base); k = int(np.argmax(seg))
    return lo + k, float(seg[k])

# ★ 고친 폭 — 봉우리 연속 확장이 아니라 **창 안 첫~마지막 임계 교차**.
#   Q–R–S 3상의 0 교차에서 끊기지 않는다(Q4-K 버그 ②).
def cross_span(x, lo, hi, frac, base):
    seg = np.abs(x[lo:hi] - base); amp = float(seg.max())
    if amp <= 1e-9: return lo, hi - 1, amp
    w = np.where(seg > frac * amp)[0]
    return lo + int(w[0]), lo + int(w[-1]), amp

# ★★★ Q4-L 의 **과잉 수정**을 되돌린다 — 첫~마지막 교차는 **다상(QRS)** 에만 옳다.
#   P·T 는 **단상**이라 봉우리에서 바깥으로 **연속 확장**하는 게 맞다.
#   Q4-L 의 P 폭 138.9ms 는 창(189ms) 전체를 P 로 잡은 결과였다.
def peak_span(x, lo, hi, frac, base):
    seg = np.abs(x[lo:hi] - base); k = int(np.argmax(seg)); amp = float(seg[k])
    if amp <= 1e-9: return lo, hi - 1, amp
    m = seg > frac * amp
    i = k
    while i > 0 and m[i - 1]: i -= 1
    j = k
    while j < len(m) - 1 and m[j + 1]: j += 1
    return lo + i, lo + j, amp

def cross_span_batch(B, lo, hi, frac, base):
    seg = np.abs(B[:, :, lo:hi] - base[:, :, None])
    amp = seg.max(-1)
    m = seg > frac * amp[..., None]
    first = m.argmax(-1)
    last = m.shape[-1] - 1 - m[:, :, ::-1].argmax(-1)
    w = np.where(m.any(-1), last - first + 1, 0)
    return w.astype(float), amp, first + lo, last + lo

def peak_span_batch(B, lo, hi, frac, base):
    """단상(P·T)용 — 봉우리에서 바깥으로 **연속** 확장. 창 전체를 삼키지 않는다."""
    seg = np.abs(B[:, :, lo:hi] - base[:, :, None])
    amp = seg.max(-1); pk = seg.argmax(-1)
    m = seg > frac * amp[..., None]
    n, nl, W = m.shape
    ar = np.arange(W)[None, None, :]
    below = ~m
    left = np.where(below & (ar < pk[..., None]), ar, -1).max(-1) + 1
    right = np.where(below & (ar > pk[..., None]), ar, W).min(-1) - 1
    w = np.clip(right - left + 1, 0, W)
    return w.astype(float), amp, left + lo, right + lo

INAMES = ["pr_rel", "dpr_ms", "pr_dev_ms", "pp_over_rr", "pw_over_qrsw", "pw_over_tw",
          "tp_resid_z", "pw_rel", "p_corr_abs", "p_polarity", "p_amp_rel", "p_found"]
MNAMES = ["corr_qrs_min", "corr_qrs_mean", "corr_full_min", "corr_st_min",
          "qrs_width", "amp_ratio", "area_ratio", "p_energy_ratio"]
PNAMES = ["corr_qrs_l0", "corr_qrs_l1", "p_e_early", "p_e_late", "corr_st_early", "corr_t_peak"]
DELIN, TMPL, P_THR = {}, {}, {}

def build_templates():
    for r in RS:
        ii = IDX_ALL[int(r)]
        okm = (REL[ii] >= TMPL_LO) & (REL[ii] <= TMPL_HI)
        if int(okm.sum()) < TMPL_MIN: okm = np.ones(len(ii), bool)
        T = np.median(BEAT[ii][okm], axis=0).astype(float)
        iso = np.median(T[:, W_P_S[0]:W_P_S[0] + 8], axis=-1)
        # ★ 리드 분리 — QRS 는 진폭 최대, **P 는 P 창 진폭 최대**(Q4-K 버그 ④)
        # ★★★ **QRS 를 먼저** 잡고, P 창을 **q_on 앞으로 자동 제한**한다.
        #   Q4-M 1차 실행이 P 폭 33.3ms · PR 40.3ms 로 죽은 원인이 이것이다 —
        #   고정 창 (20, 88) 이 q_on(≈86) 을 물어 **QRS 상승부를 P 봉우리로 잡았다**
        #   (QRS 진폭은 P 의 ~10배다). 합성 확인: 봉우리 idx 87 → 45, PR 8.3 → 152.8ms.
        lq = int(np.argmax([peak_of(T[l], W_Q_S[0], W_Q_S[1], iso[l])[1] for l in range(NL)]))
        q_on, q_off, q_amp = cross_span(T[lq], W_Q_S[0], W_Q_S[1], FRAC_QRS, iso[lq])
        p_hi = int(max(W_P_S[0] + 14, min(W_P_S[1], q_on - P_GAP)))
        lp = int(np.argmax([peak_of(T[l], W_P_S[0], p_hi, iso[l])[1] for l in range(NL)]))
        p_on, p_off, p_amp = peak_span(T[lp], W_P_S[0], p_hi, FRAC_P, iso[lp])       # 단상
        t_on, t_off, t_amp = peak_span(T[lq], W_T_S[0], W_T_S[1], FRAC_T, iso[lq])   # 단상
        p_pk = peak_of(T[lp], W_P_S[0], p_hi, iso[lp])[0]
        TMPL[int(r)] = T
        DELIN[int(r)] = dict(lq=lq, lp=lp, iso=iso, p_hi=p_hi, q=(q_on, q_off),
                             p=(p_on, p_off, p_pk),
                             t=(t_on, t_off), p_amp=p_amp, q_amp=q_amp,
                             qrs_ms=(q_off - q_on + 1) / FS * 1000.0,
                             pw_ms=(p_off - p_on + 1) / FS * 1000.0,
                             pr_ms=(q_on - p_on) / FS * 1000.0)
build_templates()
QRS_MS = float(np.median([DELIN[int(r)]["qrs_ms"] for r in RS]))
PW_MS  = float(np.median([DELIN[int(r)]["pw_ms"] for r in RS]))
PR_MS  = float(np.median([DELIN[int(r)]["pr_ms"] for r in RS]))
LP_NE_LQ = float(np.mean([DELIN[int(r)]["lp"] != DELIN[int(r)]["lq"] for r in RS]))
run.log(f"  ★ 고친 자 — QRS 폭 중앙 **{QRS_MS:.1f}ms**(Q4-K {REF['q4k']['qrs_ms']}) · "
        f"P 폭 **{PW_MS:.1f}ms**(Q4-K {REF['q4k']['pw_ms']}) · "
        f"PR(P시작→QRS시작) **{PR_MS:.1f}ms**(Q4-K {REF['q4k']['pr_ms']})")
run.log(f"  ★ 리드 분리 — P 리드 ≠ QRS 리드인 레코드 **{LP_NE_LQ:.1%}**")
Q_ON = float(np.median([DELIN[int(r)]["q"][0] for r in RS]))
P_PK = float(np.median([DELIN[int(r)]["p"][2] for r in RS]))
P_ON = float(np.median([DELIN[int(r)]["p"][0] for r in RS]))
P_HI = float(np.median([DELIN[int(r)]["p_hi"] for r in RS]))
BITE = float(np.mean([DELIN[int(r)]["p"][2] > DELIN[int(r)]["q"][0] - 8 for r in RS]))
run.log(f"  ★ **기하 진단**(표본 index) — QRS 시작 q_on 중앙 **{Q_ON:.0f}** · P 창 상단 "
        f"{P_HI:.0f}(=q_on−{P_GAP}) · P 봉우리 **{P_PK:.0f}** · P 시작 {P_ON:.0f}")
run.log(f"    **P 봉우리가 QRS 시작 8표본 안에 든 레코드 {BITE:.1%}** — 높으면 창이 QRS 를 "
        f"물은 것이다(Q4-M 1차 실행이 그래서 죽었다: P 폭 33.3ms · PR 40.3ms)")
run.log(f"    ▸ 참고 — `morph` 의 `p_energy_ratio` 창은 (25, 75) 고정이다. q_on {Q_ON:.0f} "
        f"보다 {'앞이라 안전' if Q_ON >= 76 else '**뒤라 QRS 를 물 수 있다**'}")

def interval_feats():
    out = np.zeros((len(K), 12), float)
    for r in RS:
        ii = IDX_ALL[int(r)]; d = DELIN[int(r)]; T = TMPL[int(r)]
        lp, lq = d["lp"], d["lq"]
        p_on, p_off, p_pk = d["p"]; q_on, q_off = d["q"]; t_on, t_off = d["t"]
        p_hi = d["p_hi"]
        B = BEAT[ii]
        pw0 = max(3, p_off - p_on + 1)
        seg_t = T[lp, p_on:p_on + pw0][None, :]
        best_a = np.full(len(ii), -1.0); best_s = np.ones(len(ii)); best_l = np.zeros(len(ii), int)
        for lag in range(-LAG, LAG + 1):
            a, b = p_on + lag, p_on + lag + pw0
            if a < 0 or b > p_hi: continue          # ★ QRS 를 물지 않게 클램프
            c = corr_to(B[:, lp, a:b], seg_t)
            ac = np.abs(c)                        # ★★★ 역위 P 를 잡는다(Q4-K 버그 ①)
            upd = ac > best_a
            best_a[upd] = ac[upd]; best_s[upd] = np.sign(c[upd]); best_l[upd] = lag
        # ★★★ 검출 문턱을 **레코드별 영점으로 교정**한다.
        #   61개 지연 중 최대 |corr| 는 **선택 효과**로 잡음에서도 0.5 를 쉽게 넘는다.
        #   그래서 **표본 치환 템플릿**(모양은 완전히 파괴, 진폭분포는 보존)으로 같은 절차를
        #   돌려 그 95 백분위를 문턱으로 쓴다. `p_found` 가 그제서야 의미를 갖는다.
        #   ⚠️ 시간 역전은 안 쓴다 — **대칭 파형에서는 항등**이라 영점이 신호와 같아진다.
        seg_r = seg_t[:, np.random.RandomState(SEED0 + int(r)).permutation(pw0)]
        nul_a = np.full(len(ii), -1.0)
        for lag in range(-LAG, LAG + 1):
            a, b = p_on + lag, p_on + lag + pw0
            if a < 0 or b > p_hi: continue          # ★ 영점도 같은 창에서
            nul_a = np.maximum(nul_a, np.abs(corr_to(B[:, lp, a:b], seg_r)))
        thr_r = max(P_CORR_MIN, float(np.quantile(nul_a, P_NULL_Q)))
        P_THR[int(r)] = thr_r
        p_found = (best_a >= thr_r).astype(float)
        base_b = np.median(B[:, :, W_P_S[0]:W_P_S[0] + 8], axis=-1)
        pw, pa, p_first, _ = peak_span_batch(B, max(0, p_on - LAG),
                                             min(p_hi, p_off + LAG + 1), FRAC_P, base_b)
        qw, qa, q_first, _ = cross_span_batch(B, W_Q_S[0], W_Q_S[1], FRAC_QRS, base_b)
        tw, ta, _, t_last = peak_span_batch(B, W_T_S[0], W_T_S[1], FRAC_T, base_b)
        pw = pw[:, lp]; pa = pa[:, lp]
        qw = qw[:, lq]; tw = tw[:, lq]
        p_on_b = (p_on + best_l).astype(float)
        q_on_b = q_first[:, lq].astype(float)
        pr_ms = (q_on_b - p_on_b) / FS * 1000.0      # ★ P 시작 → QRS 시작
        ok_ = p_found > 0
        med_pr = float(np.median(pr_ms[ok_])) if ok_.any() else float(np.median(pr_ms))
        dpr = np.r_[0.0, np.diff(pr_ms)]
        rr_s = pre[ii] * FS
        t_last_prev = np.r_[float(t_off), t_last[:-1, lq].astype(float)]
        tp = rr_s + p_on_b - t_last_prev
        A = np.c_[np.ones(len(ii)), rr_s]
        coef, *_ = np.linalg.lstsq(A, tp, rcond=None)
        tp_res = tp - A @ coef
        med_pw = float(np.median(pw)) + 1e-9; med_pa = float(np.median(pa)) + 1e-9
        out[ii, 0] = pr_ms / (med_pr + 1e-9)
        out[ii, 1] = dpr
        out[ii, 2] = pr_ms - med_pr
        out[ii, 3] = 1.0 - (dpr / 1000.0 * FS) / (rr_s + 1e-9)
        out[ii, 4] = pw / (qw + 1e-9)
        out[ii, 5] = pw / (tw + 1e-9)
        out[ii, 6] = tp_res / (float(np.std(tp_res)) + 1e-9)
        out[ii, 7] = pw / med_pw
        out[ii, 8] = best_a
        out[ii, 9] = best_s                          # ★★★ 역위 P = 소견
        out[ii, 10] = pa / med_pa
        out[ii, 11] = p_found
    return np.nan_to_num(out, nan=0.0, posinf=0.0, neginf=0.0)

def morph_feats(extended=False):
    ncol = 6 if extended else 8
    out = np.zeros((len(K), ncol), float)
    WQ, WF, WS, WP, WW = (85, 125), (60, 220), (130, 260), (25, 75), (80, 130)
    WSE, WTP, WPE, WPL = (128, 175), (175, 225), (25, 50), (50, 75)
    for r in RS:
        ii = IDX_ALL[int(r)]; B = BEAT[ii]; T = TMPL[int(r)]
        if not extended:
            cq = corr_to(B[:, :, WQ[0]:WQ[1]], T[:, WQ[0]:WQ[1]])
            cf = corr_to(B[:, :, WF[0]:WF[1]], T[:, WF[0]:WF[1]])
            cs = corr_to(B[:, :, WS[0]:WS[1]], T[:, WS[0]:WS[1]])
            seg = B[:, :, WW[0]:WW[1]]; med = np.median(seg, axis=-1, keepdims=True)
            amp = np.abs(seg - med).max(-1, keepdims=True) + 1e-9
            wid = (np.abs(seg - med) > 0.5 * amp).mean(-1)
            q = B[:, :, WQ[0]:WQ[1]]; ptp = q.max(-1) - q.min(-1)
            tq = T[:, WQ[0]:WQ[1]]; tptp = float(np.mean(tq.max(-1) - tq.min(-1))) + 1e-9
            area = np.abs(q - np.median(q, axis=-1, keepdims=True)).sum(-1)
            tarea = float(np.mean(np.abs(tq - np.median(tq, axis=-1, keepdims=True)).sum(-1))) + 1e-9
            p = B[:, :, WP[0]:WP[1]]
            pe = np.sqrt(((p - p.mean(-1, keepdims=True)) ** 2).mean(-1))
            tp_ = T[:, WP[0]:WP[1]]
            tpe = float(np.mean(np.sqrt(((tp_ - tp_.mean(-1, keepdims=True)) ** 2).mean(-1)))) + 1e-9
            out[ii, 0] = cq.min(1); out[ii, 1] = cq.mean(1); out[ii, 2] = cf.min(1)
            out[ii, 3] = cs.min(1); out[ii, 4] = wid.mean(1)
            out[ii, 5] = ptp.mean(1) / tptp; out[ii, 6] = area.mean(1) / tarea
            out[ii, 7] = pe.mean(1) / tpe
        else:
            cq = corr_to(B[:, :, WQ[0]:WQ[1]], T[:, WQ[0]:WQ[1]])
            out[ii, 0] = cq[:, 0]; out[ii, 1] = cq[:, min(1, NL - 1)]
            for c_, W_ in ((2, WPE), (3, WPL)):
                p = B[:, :, W_[0]:W_[1]]
                pe = np.sqrt(((p - p.mean(-1, keepdims=True)) ** 2).mean(-1))
                tp_ = T[:, W_[0]:W_[1]]
                tpe = float(np.mean(np.sqrt(((tp_ - tp_.mean(-1, keepdims=True)) ** 2)
                                            .mean(-1)))) + 1e-9
                out[ii, c_] = pe.mean(1) / tpe
            out[ii, 4] = corr_to(B[:, :, WSE[0]:WSE[1]], T[:, WSE[0]:WSE[1]]).min(1)
            out[ii, 5] = corr_to(B[:, :, WTP[0]:WTP[1]], T[:, WTP[0]:WTP[1]]).min(1)
    return np.nan_to_num(out, nan=0.0, posinf=0.0, neginf=0.0)

# ── ★★★ 축 A — P-on-T: **직전 박동의 T 창**을 이 박동의 특징으로
PONAMES = ["prevT_corr", "prevT_energy", "prevT_late", "prevT_resid_max",
           "prevT_resid_pos", "prevTP_energy"]
def pont_feats():
    WT, WTL = W_T_S, (max(W_T_S[0], W_T_S[1] - 45), W_T_S[1])
    WTP = (min(LW - 12, W_T_S[1]), LW)
    out = np.zeros((len(K), 6), float)
    for r in RS:
        ii = IDX_ALL[int(r)]; B = BEAT[ii]; T = TMPL[int(r)]; lq = DELIN[int(r)]["lq"]
        seg = B[:, lq, WT[0]:WT[1]]; tseg = T[lq, WT[0]:WT[1]]
        c = corr_to(seg, tseg[None, :])
        e = np.sqrt(((seg - seg.mean(-1, keepdims=True)) ** 2).mean(-1))
        te = float(np.sqrt(((tseg - tseg.mean()) ** 2).mean())) + 1e-9
        sl = B[:, lq, WTL[0]:WTL[1]]; tsl = T[lq, WTL[0]:WTL[1]]
        el = np.sqrt(((sl - sl.mean(-1, keepdims=True)) ** 2).mean(-1))
        tel = float(np.sqrt(((tsl - tsl.mean()) ** 2).mean())) + 1e-9
        res = np.abs(seg - tseg[None, :])
        amp = float(np.ptp(tseg)) + 1e-9
        rmax = res.max(-1) / amp
        rpos = res.argmax(-1) / max(1, res.shape[-1] - 1)
        sp = B[:, lq, WTP[0]:WTP[1]]; tsp = T[lq, WTP[0]:WTP[1]]
        ep = np.sqrt(((sp - sp.mean(-1, keepdims=True)) ** 2).mean(-1))
        tep = float(np.sqrt(((tsp - tsp.mean()) ** 2).mean())) + 1e-9
        # ★ **직전 박동**의 값을 이 박동에 붙인다(첫 박동은 자기 값으로 채운다)
        sh = lambda v: np.r_[v[0], v[:-1]]
        out[ii, 0] = sh(c); out[ii, 1] = sh(e / te); out[ii, 2] = sh(el / tel)
        out[ii, 3] = sh(rmax); out[ii, 4] = sh(rpos); out[ii, 5] = sh(ep / tep)
    return np.nan_to_num(out, nan=0.0, posinf=0.0, neginf=0.0)

# ── ★★ 축 B — 레코드 내 **무라벨** 군집(개인화): 「이 환자의 PAC 은 서로 닮았다」
CLNAMES = ["clus_minor", "clus_size_frac", "clus_d_major", "clus_d_own"]
def clus_feats(M):
    from sklearn.cluster import KMeans
    out = np.zeros((len(K), 4), float)
    for r in RS:
        ii = IDX_ALL[int(r)]
        Z = M[ii]; Z = (Z - Z.mean(0)) / (Z.std(0) + 1e-9)
        nc = int(min(N_CLUS, max(2, len(ii) // 50)))
        km = KMeans(n_clusters=nc, n_init=4, random_state=SEED0).fit(Z)
        lab = km.labels_; cnt = np.bincount(lab, minlength=nc)
        major = int(np.argmax(cnt)); minor = int(np.argmin(cnt))
        d_all = np.linalg.norm(Z[:, None, :] - km.cluster_centers_[None, :, :], axis=-1)
        sd = float(np.median(d_all)) + 1e-9
        out[ii, 0] = (lab == minor).astype(float)
        out[ii, 1] = cnt[lab] / float(len(ii))
        out[ii, 2] = d_all[:, major] / sd
        out[ii, 3] = d_all[np.arange(len(ii)), lab] / sd
    return np.nan_to_num(out, nan=0.0, posinf=0.0, neginf=0.0)

T_FEAT = time.time()
MORPH = morph_feats(False); MORPHX = morph_feats(True); INTV = interval_feats()
PONT = pont_feats(); CLUS = clus_feats(np.c_[MORPH, INTV])
run.log(f"  ({time.time()-T_FEAT:.0f}초) 형태 8 + 구간 12 + **P-on-T 6** + **군집 4**열")
run.log(f"  ★★★ P-on-T — " + " · ".join(PONAMES))
run.log(f"  ★★ 군집(무라벨 · 레코드별 KMeans k={N_CLUS}) — " + " · ".join(CLNAMES))

# ── ★★★ P0 자 검증 — 하나라도 깨지면 중단(R16)
POL = INTV[:, 9]; PABS = INTV[:, 8]; PF = INTV[:, 11]
is_v = np.isin(SYM, ("V", "E", "F")); is_n = np.isin(SYM, ("N", "L", "R", "e", "j", "n"))
pv = float(np.mean(PABS[is_v])) if is_v.any() else float("nan")
pn = float(np.mean(PABS[is_n])) if is_n.any() else float("nan")
inv_rate = float(np.mean(POL < 0))
fv = float(np.mean(PF[is_v])) if is_v.any() else float("nan")
fn = float(np.mean(PF[is_n])) if is_n.any() else float("nan")
THR_MED = float(np.median(list(P_THR.values())))
run.log(f"\n  ★★★ **P0 자 검증**(`sym` 은 **자 검증에만** 쓴다 — 적합·선택엔 안 쓴다)")
run.log(f"    QRS 폭 {QRS_MS:.1f}ms  범위 [{QRS_LO_MS}, {QRS_HI_MS}]  "
        + ("✅" if QRS_LO_MS <= QRS_MS <= QRS_HI_MS else "❌"))
run.log(f"    PR    {PR_MS:.1f}ms  범위 [{PR_LO_MS}, {PR_HI_MS}]  "
        + ("✅" if PR_LO_MS <= PR_MS <= PR_HI_MS else "❌"))
run.log(f"    **역위 P 비율 {inv_rate:.1%}** — 0 이면 `|corr|` 정렬이 작동 안 하는 것이다")
run.log(f"    **P 검출 타당성** |corr| — N 박동 {pn:.4f} vs **V 박동 {pv:.4f}** "
        + ("✅ V 가 낮다" if np.isfinite(pv) and pv < pn else "❌ V 가 안 낮다")
        + f"  (전체 검출률 {float(PF.mean()):.1%} · Q4-K {REF['q4k']['p_found']:.1%})")
run.log(f"    **검출률** — N 박동 {fn:.1%} vs **V 박동 {fv:.1%}** "
        + ("✅ V 가 낮다" if np.isfinite(fv) and fv < fn else "❌ V 가 안 낮다")
        + f"  · 레코드별 영점교정 문턱 중앙 **{THR_MED:.3f}**(표본치환 템플릿 {P_NULL_Q:.0%} 분위)")
_fail = []
if not (QRS_LO_MS <= QRS_MS <= QRS_HI_MS): _fail.append(f"QRS 폭 {QRS_MS:.1f}ms")
if not (PR_LO_MS <= PR_MS <= PR_HI_MS): _fail.append(f"PR {PR_MS:.1f}ms")
if not (PW_LO_MS <= PW_MS <= PW_HI_MS): _fail.append(f"P 폭 {PW_MS:.1f}ms(Q4-L 138.9)")
if not (inv_rate > 0.001): _fail.append("역위 P 0%")
if not (np.isfinite(pv) and np.isfinite(pn) and pv < pn):
    _fail.append(f"V |corr| {pv:.3f} >= N {pn:.3f}")
if not (np.isfinite(fv) and np.isfinite(fn) and fv < fn):
    _fail.append(f"V 검출률 {fv:.1%} >= N {fn:.1%} — 검출기가 P 유무를 반영 못 한다")
if _fail:
    raise AssetError("P0 자 검증 실패 — " + " · ".join(_fail)
                     + " ⇒ 고장난 자로 잰 결과는 무효다. 구간 축을 **접는다**(R16 · R35 ①)")

_rs = np.random.RandomState(SEED0 + 7)
def rec_shuffle(M):
    S_ = M.copy()
    for r in RS:
        ii = IDX_ALL[int(r)]; S_[ii] = M[ii][_rs.permutation(len(ii))]
    return S_
P_SH = rec_shuffle(PONT); C_SH = rec_shuffle(CLUS)
_moved = float(np.mean(np.any(np.abs(P_SH - PONT) > 1e-12, axis=1)))
if _moved < 0.5: raise AssetError(f"대조군이 거의 항등이다({_moved:.3f})(R35 ①)")
FEAT = {"base": F_BASE, "morph": np.c_[F_BASE, MORPH],
        "pont": np.c_[F_BASE, MORPH, PONT], "pshuf": np.c_[F_BASE, MORPH, P_SH],
        "clus": np.c_[F_BASE, MORPH, CLUS], "cshuf": np.c_[F_BASE, MORPH, C_SH],
        "comb": np.c_[F_BASE, MORPH, PONT, CLUS]}
for a, b in (("pont", "pshuf"), ("clus", "cshuf")):
    if FEAT[a].shape[1] != FEAT[b].shape[1]: raise AssetError(f"차원 대조군 불일치 {a}/{b}")
run.log("  차원 — " + " · ".join(f"{a} {FEAT[a].shape[1]}" for a in ARMS)
        + f" · 대조군 이동 {_moved:.1%}")

IDXS = {int(r): IDX_ALL[int(r)] for r in RS}
REC_OK = [int(r) for r in RS
          if TT_[IDXS[int(r)]].sum() >= MIN_S and (~TT_[IDXS[int(r)]]).sum() >= MIN_N]
BURD = {r: float(TT_[IDXS[r]].mean()) for r in REC_OK}
NS_ = {r: int(TT_[IDXS[r]].sum()) for r in REC_OK}
NRE = len(REC_OK)
run.log(f"  레코드 {len(RS)} · 채점 가능 **{NRE}**")

SLOPES, SLOPE_BY, _CUR = [], {}, [None]
def make_cal(s, y):
    lr = LogisticRegression(max_iter=3000, C=1e6).fit(np.asarray(s).reshape(-1, 1),
                                                       np.asarray(y).astype(int))
    a, b = float(lr.coef_[0, 0]), float(lr.intercept_[0])
    SLOPES.append(a)
    if _CUR[0] is not None: SLOPE_BY.setdefault(_CUR[0], []).append(a)
    return lambda v: a * np.asarray(v, float) + b

def split_rest(held):
    rest = sorted([r for r in REC_OK if r != held], key=lambda r: (BURD[r], r))
    dv = [r for i, r in enumerate(rest) if i % DEV_EVERY == 0]
    return [r for r in rest if r not in set(dv)], dv

def fit_fold(X, held, y_override):
    tr_r, dv_r = split_rest(held)
    tr = np.concatenate([IDXS[r] for r in tr_r]); dv = np.concatenate([IDXS[r] for r in dv_r])
    te = IDXS[held]
    Ftr = X[tr]; fmu = Ftr.mean(0); fsd = Ftr.std(0) + 1e-9
    ytr = TT_[tr].astype(int) if y_override is None else np.asarray(y_override[held], int)
    lr = LogisticRegression(max_iter=3000, C=1.0).fit((Ftr - fmu) / fsd, ytr)
    f = lambda ii: lr.decision_function((X[ii] - fmu) / fsd)
    raw = f(te)
    return te, make_cal(f(dv), TT_[dv])(raw), raw

def loro(X, y_override=None, tag=None):
    out = np.full(len(K), np.nan); raw = np.full(len(K), np.nan); _CUR[0] = tag
    for held in REC_OK:
        te, v, rv = fit_fold(X, held, y_override)
        out[te] = v; raw[te] = rv
    return out, raw

def per_auc(L):
    return {r: float(roc_auc_score(TT_[IDXS[r]].astype(int), L[IDXS[r]])) for r in REC_OK}
def per_ap(L):
    return {r: float(average_precision_score(TT_[IDXS[r]].astype(int), L[IDXS[r]]))
            for r in REC_OK}
def tp_at(L, r, k):
    idx = IDXS[r]; sc = L[idx]; yy = TT_[idx]
    k = int(min(max(1, k), len(idx)))
    fl = sc >= np.partition(sc, -k)[-k]
    return int((fl & yy).sum()), fl
# ★ 항등식 — 달성률 = TP / min(S, k). k <= S 면 **정밀도@k**, k > S 면 재현율
def ach_at(L, r, k):
    tp, _ = tp_at(L, r, k)
    return tp / max(1, min(NS_[r], int(k)))
def per_ksw(L):
    return {r: float(np.mean([ach_at(L, r, k) for k in K_SWEEP])) for r in REC_OK}
def per_ach(L):
    return {r: ach_at(L, r, MAIN_K) for r in REC_OK}
CONFIG["cohort"] = dict(n_ok=NRE, moved=_moved, qrs_ms=QRS_MS, pw_ms=PW_MS, pr_ms=PR_MS,
                        inv_rate=inv_rate, p_found=float(PF.mean()), p_corr_v=pv, p_corr_n=pn,
                        lead_split=LP_NE_LQ, p_found_v=fv, p_found_n=fn, p_thr=THR_MED,
                        q_on=Q_ON, p_pk=P_PK, p_on=P_ON, p_hi=P_HI, bite=BITE,
                        dims={a: int(FEAT[a].shape[1]) for a in ARMS})
run.save_json("config", CONFIG)


In [ ]:
# CELL 3 — 【P-A】 ★★★ P1 중복 감사 v2(레코드 내 순위) · 실행
run.log("\n" + "=" * 100)
run.log("【P-A】 ★★★ **P1 중복 감사 v2** — 풀링·선형이 아니라 **레코드 내 순위**로 잰다")
run.log("=" * 100)
_ZB = (F_BASE - F_BASE.mean(0)) / (F_BASE.std(0) + 1e-9)
# 레코드 내 스피어만 |ρ| 의 base 열 최댓값 — **단조 변환에 불변**
def rank_dup(col):
    best = 0.0
    for j in range(F_BASE.shape[1]):
        v = []
        for r in REC_OK:
            ii = IDXS[r]
            if np.std(col[ii]) < 1e-12 or np.std(F_BASE[ii, j]) < 1e-12: continue
            rr = spearmanr(col[ii], F_BASE[ii, j]).statistic
            if np.isfinite(rr): v.append(abs(float(rr)))
        if v: best = max(best, float(np.median(v)))
    return best
def pooled_r2(col):
    y = np.asarray(col, float)
    if np.std(y) < 1e-12: return 1.0
    return float(max(0.0, min(1.0, LinearRegression().fit(_ZB, y).score(_ZB, y))))
def within_var(col):
    y = np.asarray(col, float); tot = np.std(y) + 1e-12
    return float(np.mean([np.std(y[IDXS[r]]) for r in REC_OK]) / tot)
def uni_one(col):
    a_ = []
    for r in REC_OK:
        ii = IDXS[r]; yy = TT_[ii].astype(int); v = col[ii]
        if 0 < yy.sum() < len(ii) and np.std(v) > 0:
            a_.append(abs(roc_auc_score(yy, v) - 0.5))
    return float(np.mean(a_)) if a_ else 0.0

run.log(f"  {'열':<15}{'블록':>6}{'레코드내|ρ|':>12}{'풀링R²':>9}{'분산비':>8}{'단변량':>9}{'판정':>10}")
DUP = {}
for blk, M, names in (("형태", MORPH, MNAMES), ("구간", INTV, INAMES),
                      ("PonT", PONT, PONAMES), ("군집", CLUS, CLNAMES)):
    for j, nm in enumerate(names):
        rd = rank_dup(M[:, j]); r2 = pooled_r2(M[:, j])
        wv = within_var(M[:, j]); u = uni_one(M[:, j])
        vd = "⛔ 중복" if rd > DUP_RHO else ("⚠️ 레코드상수" if wv < 0.15 else "✅ 새 축")
        DUP[nm] = dict(block=blk, rho=rd, r2=r2, within=wv, uni=u, verdict=vd)
        run.log(f"  {nm:<15}{blk:>6}{rd:>12.4f}{r2:>9.4f}{wv:>8.3f}{u:>9.4f}{vd:>10}")
n_dup = sum(1 for v in DUP.values() if v["verdict"].startswith("⛔"))
run.log(f"  ⇒ 중복 **{n_dup}** / {len(DUP)}열 (문턱 레코드내 |ρ| > {DUP_RHO})")

rr_s_all = pre * FS
tp_k = np.zeros(len(K))
for r in RS:
    ii = IDX_ALL[int(r)]; d = DELIN[int(r)]
    tp_k[ii] = 1.0 + (d["p"][0] - d["t"][1]) / (rr_s_all[ii] + 1e-9)
rd_k, r2_k, u_k = rank_dup(tp_k), pooled_r2(tp_k), uni_one(tp_k)
run.log(f"\n  ★★★ **회고 검증** — Q4-K 의 `tp_over_rr` 를 새 자로 다시 잰다")
run.log(f"    레코드내 |ρ| **{rd_k:.4f}** · 풀링 R² {r2_k:.4f}(Q4-K {REF['q4k']['tp_r2']}) · "
        f"단변량 {u_k:.4f}(Q4-K {REF['q4k']['tp_uni']}) → "
        + ("**⛔ 중복으로 잡힌다 — 새 자가 작동한다**" if rd_k > DUP_RHO
           else "⚠️ 새 자로도 중복이 아니다 — 진단을 재검토해야 한다"))
g_("P1", "(입구 검사)",
   f"중복 {n_dup}/{len(DUP)} · 회고 `tp_over_rr` 레코드내|ρ| {rd_k:.4f} vs 풀링 R² {r2_k:.4f} · "
   f"PonT 최고 단변량 {max(DUP[n]['uni'] for n in PONAMES):.4f} · 군집 최고 "
   f"{max(DUP[n]['uni'] for n in CLNAMES):.4f} · 형태 최고 "
   f"{max(DUP[n]['uni'] for n in MNAMES):.4f}")

run.log("\n" + "=" * 100)
run.log("【P-B】 실행")
run.log("=" * 100)
T0 = time.time()
L, LRAW = {}, {}
for a in ARMS:
    L[a], LRAW[a] = loro(FEAT[a], None, tag=a)
    run.log(f"  ({time.time()-T0:>5.0f}초) {a} 완료")
AUC = {a: per_auc(L[a]) for a in ARMS}
AP = {a: per_ap(L[a]) for a in ARMS}
KSW = {a: per_ksw(L[a]) for a in ARMS}
ACH = {a: per_ach(L[a]) for a in ARMS}
CAL_GAP = max(abs(AUC[a][r] - per_auc(LRAW[a])[r]) for a in ARMS for r in REC_OK)
bad_arms = []
for a in ARMS:
    sa = np.array(SLOPE_BY.get(a, []), float); mac = float(np.mean(list(AUC[a].values())))
    if len(sa) and mac > MIN_AUC_SLOPE and (np.median(sa) <= 0 or np.mean(sa <= 0) > MAX_NEG_SLOPE):
        bad_arms.append(a)
if bad_arms: raise AssetError(f"P0 실패 — 체계적 반전 {bad_arms}(R29 ②)")
g_("P0", "✅ 지지",
   f"QRS {QRS_MS:.1f}ms · PR {PR_MS:.1f}ms · 역위 P {inv_rate:.1%} · |corr| N {pn:.3f} > V "
   f"{pv:.3f} · 검출률 N {fn:.1%} > V {fv:.1%} · 문턱 {THR_MED:.3f} · 리드분리 {LP_NE_LQ:.1%}")

run.log(f"\n  {'팔':<9}{'차원':>5}{'k-스윕★':>10}{'달성률@300':>12}{'AUROC':>9}{'PR-AUC':>9}")
for a in ARMS:
    run.log(f"  {a:<9}{FEAT[a].shape[1]:>5}{np.mean(list(KSW[a].values())):>10.4f}"
            f"{np.mean(list(ACH[a].values())):>12.4f}{np.mean(list(AUC[a].values())):>9.4f}"
            f"{np.mean(list(AP[a].values())):>9.4f}")
run.log(f"  (Q4-K 앵커 달성률@300 — base {REF['q4k']['base']['ach']} · morph "
        f"{REF['q4k']['morph']['ach']} · intv {REF['q4k']['intv']['ach']})")
run.log(f"  ▸ k 별 달성률(base/morph/comb) — " + " · ".join(
    f"k={k} {np.mean([ach_at(L['base'], r, k) for r in REC_OK]):.3f}/"
    f"{np.mean([ach_at(L['morph'], r, k) for r in REC_OK]):.3f}/"
    f"{np.mean([ach_at(L['comb'], r, k) for r in REC_OK]):.3f}" for k in K_SWEEP))
CONFIG["P0"] = dict(qrs_ms=QRS_MS, pr_ms=PR_MS, pw_ms=PW_MS, inv_rate=inv_rate,
                    p_corr_v=pv, p_corr_n=pn, cal_gap=float(CAL_GAP), lead_split=LP_NE_LQ,
                    ksw={a: float(np.mean(list(KSW[a].values()))) for a in ARMS},
                    ach={a: float(np.mean(list(ACH[a].values()))) for a in ARMS},
                    auc={a: float(np.mean(list(AUC[a].values()))) for a in ARMS},
                    ap={a: float(np.mean(list(AP[a].values()))) for a in ARMS})
CONFIG["P1"] = dict(dup=DUP, retro=dict(rho=rd_k, r2=r2_k, uni=u_k))
run.save_json("config", CONFIG)


In [ ]:
# CELL 4 — 【P-C】 영점 · ★★★ P2 주 관문 · P3 · P5
run.log("\n" + "=" * 100)
run.log("【P-C】 영점(raw · rep 병기) · ★★★ **P2 — 고친 구간이 k-스윕을 올리는가**")
run.log("=" * 100)
SRC = {"ksw": KSW, "auc": AUC, "ach": ACH}
NUL = {c[0]: {k: {r: [] for r in REC_OK} for k in ("ksw", "auc")} for c in CONTRASTS}
REPM = {c[0]: {"ksw": [], "auc": []} for c in CONTRASTS}
for s_ in range(N_PERM):
    rr = np.random.RandomState(SEED0 + 400 + s_)
    yov = {}
    for held in REC_OK:
        tr_r, _ = split_rest(held)
        tr = np.concatenate([IDXS[r] for r in tr_r])
        yov[held] = TT_[tr].astype(int)[rr.permutation(len(tr))]
    RW = {a: loro(FEAT[a], yov)[1] for a in ARMS}
    Sa = {"auc": {a: per_auc(RW[a]) for a in ARMS}, "ksw": {a: per_ksw(RW[a]) for a in ARMS}}
    for nm, x_, y_ in CONTRASTS:
        for key in ("ksw", "auc"):
            d = [Sa[key][y_][r] - Sa[key][x_][r] for r in REC_OK]
            REPM[nm][key].append(float(np.mean(d)))
            for i_, r in enumerate(REC_OK): NUL[nm][key][r].append(d[i_])
    run.log(f"    ({time.time()-T0:>5.0f}초) 영점 rep {s_+1}/{N_PERM}")
NSTAT, NREP = {}, {}
for nm, x_, y_ in CONTRASTS:
    NSTAT[nm], NREP[nm] = {}, {}
    for key in ("ksw", "auc"):
        NSTAT[nm][key] = boot_mean([float(np.mean(v)) for v in NUL[nm][key].values()],
                                   SEED0 + 61 + len(nm) + (0 if key == "ksw" else 7), NB_BOOT)
        v = np.array(REPM[nm][key], float)
        sd = float(v.std(ddof=1)) if len(v) > 1 else float("nan")
        se = sd / np.sqrt(max(1, len(v)))
        NREP[nm][key] = dict(mean=float(v.mean()), lo=float(v.mean() - 1.96 * se),
                             hi=float(v.mean() + 1.96 * se), n=int(len(v)))
NULL_BLUR = []
# ★ 보수적 문턱은 ✅ 를 어렵게 하는 건 맞지만 **❌ 를 만들어선 안 된다**(Q4-K 에서 넣은 규칙)
def two_verdicts(nm, key, obs):
    nhi = max(NSTAT[nm][key][2], NREP[nm][key]["hi"])
    thr = max(0.0, nhi) if np.isfinite(nhi) else float("nan")
    nh = max(mde(NSTAT[nm][key][1], NSTAT[nm][key][2]),
             mde(NREP[nm][key]["lo"], NREP[nm][key]["hi"]))
    blur = np.isfinite(nh) and np.isfinite(obs["mde"]) and nh > obs["mde"]
    def dem(v):
        if blur and v.startswith("❌"):
            NULL_BLUR.append((nm, key)); return "⚠️ 미결"
        return v
    return (dem(decide(obs["lo"], obs["hi"], thr, ">")), thr,
            dem(decide(obs["lo"], obs["hi"], nhi, ">")), nhi)

OBS, TAB = {}, {}
run.log(f"\n  {'대비':<15}{'Δ k-스윕 ★1차':>27}{'배포':>7}{'Δ AUROC(2차)':>25}{'배포':>7}")
for nm, x_, y_ in CONTRASTS:
    OBS[nm], TAB[nm] = {}, {}
    row = f"  {nm:<15}"
    for key in ("ksw", "auc"):
        m_, lo_, hi_, n_ = boot_pair([SRC[key][x_][r] for r in REC_OK],
                                     [SRC[key][y_][r] for r in REC_OK],
                                     SEED0 + 81 + len(nm) + (0 if key == "ksw" else 7), NB_BOOT)
        o = dict(mean=m_, lo=lo_, hi=hi_, n=int(n_), mde=float(mde(lo_, hi_)))
        OBS[nm][key] = o
        vd_, td_, vm_, tm_ = two_verdicts(nm, key, o)
        TAB[nm][key] = dict(obs=o, null_rec=list(NSTAT[nm][key][:3]), null_rep=NREP[nm][key],
                            thr_deploy=float(td_), thr_mech=float(tm_), v_deploy=vd_, v_mech=vm_)
        row += f"{m_:>+9.4f} [{lo_:+.4f},{hi_:+.4f}]{vd_:>7}"
    run.log(row)
if NULL_BLUR:
    run.log(f"  ⚠️ 영점 흐림으로 기각을 강등한 대비 {len(set(NULL_BLUR))}건 — "
            f"**증거 없음**이지 반대 증거가 아니다(R33 ①)")

om = OBS[MAIN_CT][PRIMARY]; vd, td, vm, tm = two_verdicts(MAIN_CT, PRIMARY, om)
oa = OBS[MAIN_CT][SECONDARY]; vd2, td2, _, _ = two_verdicts(MAIN_CT, SECONDARY, oa)
run.log(f"\n  ★★★ **P2 주 관문** — `{MAIN_CT}` 차원 {FEAT['pont'].shape[1]} 동일")
run.log(f"    **1차 k-스윕** Δ **{om['mean']:+.4f}** [{om['lo']:+.4f}, {om['hi']:+.4f}] · "
        f"영점 {NSTAT[MAIN_CT][PRIMARY][0]:+.4f} · 문턱 {td:+.4f} · MDE {om['mde']:.4f} → {vd}")
run.log(f"    2차 AUROC Δ {oa['mean']:+.4f} [{oa['lo']:+.4f}, {oa['hi']:+.4f}] → {vd2}")
g_("P2", vd,
   f"`pont` vs `pshuf` k-스윕 Δ {om['mean']:+.4f} [{om['lo']:+.4f}, {om['hi']:+.4f}] "
   f"(문턱 {td:+.4f}) · AUROC {oa['mean']:+.4f} {vd2} — "
   + ("**직전 박동의 T 창이 형태 위에 신호를 더한다**" if vd.startswith("✅") else
      ("P-on-T 도 형태 위에는 못 얹힌다 — 형태 축이 포화다(R39 ①)" if vd.startswith("❌")
       else "가르지 못했다(R33 ①)")))

v_cc = TAB["clus-cshuf"][PRIMARY]["v_deploy"]; v_cm = TAB["comb-morph"][PRIMARY]["v_deploy"]
run.log(f"\n  ★★ **P3** — `clus` vs `cshuf` {OBS['clus-cshuf'][PRIMARY]['mean']:+.4f} {v_cc} · "
        f"**`comb` vs `morph`(현재 최선 위에)** {OBS['comb-morph'][PRIMARY]['mean']:+.4f} {v_cm} · "
        f"`pont − morph` {OBS['pont-morph'][PRIMARY]['mean']:+.4f}")
g_("P3", v_cm,
   f"`comb − morph` k-스윕 {OBS['comb-morph'][PRIMARY]['mean']:+.4f} "
   f"[{OBS['comb-morph'][PRIMARY]['lo']:+.4f}, {OBS['comb-morph'][PRIMARY]['hi']:+.4f}] {v_cm} · "
   f"`clus − cshuf` {OBS['clus-cshuf'][PRIMARY]['mean']:+.4f} {v_cc} · 형태 재현 "
   f"`morph − base` {OBS['morph-base'][PRIMARY]['mean']:+.4f}(Q4-L +0.1570)")
CONFIG["P2"] = TAB
run.save_json("config", CONFIG)


In [ ]:
# CELL 5 — 【P-D】 열별 기여 · ★★ 잔차 부스팅(GPU) · 위양성 구성
run.log("\n" + "=" * 100)
run.log("【P-D】 열별 기여 · 위양성 구성")
run.log("=" * 100)
BLOCKS = {"P-on-T": ("PONT", list(range(6))), "군집": ("CLUS", list(range(4)))}
run.log(f"  {'블록':<10}{'차원':>4}{'Δ k-스윕(형태 위에)':>28}{'Δ AUROC':>22}{'최고|ρ|':>9}")
P4 = {}
for nm, (src_, cols) in BLOCKS.items():
    M = PONT if src_ == "PONT" else CLUS
    names = PONAMES if src_ == "PONT" else CLNAMES
    Lb, _ = loro(np.c_[F_BASE, MORPH, M[:, cols]], None, tag=f"blk{nm}")
    kb = per_ksw(Lb); ub = per_auc(Lb)
    dk = boot_pair([KSW["morph"][r] for r in REC_OK], [kb[r] for r in REC_OK],
                   SEED0 + 200 + len(nm), NB_BOOT)
    du = boot_pair([AUC["morph"][r] for r in REC_OK], [ub[r] for r in REC_OK],
                   SEED0 + 230 + len(nm), NB_BOOT)
    rmax = max(DUP[names[j]]["rho"] for j in cols)
    P4[nm] = dict(cols=[names[j] for j in cols], ksw=list(dk[:3]), auc=list(du[:3]),
                  rho_max=float(rmax))
    run.log(f"  {nm:<10}{len(cols):>4}{dk[0]:>+9.4f} [{dk[1]:+.4f},{dk[2]:+.4f}]"
            f"{du[0]:>+9.4f} [{du[1]:+.4f},{du[2]:+.4f}]{rmax:>9.3f}")
run.log(f"\n  ▸ **열별 단변량**(|AUROC−0.5|)")
for nm, names in (("P-on-T", PONAMES), ("군집", CLNAMES), ("형태", MNAMES)):
    run.log(f"    {nm:<8}" + " · ".join(f"{n} {DUP[n]['uni']:.4f}" for n in names))
g_("P4", "(관문 아님)",
   " · ".join(f"{n} {P4[n]['ksw'][0]:+.4f}" for n in BLOCKS)
   + f" · PonT 최고 단변량 {max(DUP[n]['uni'] for n in PONAMES):.4f}")

# ── ★★ P5 잔차 부스팅 — logit = logit_cpu(고정) + α·CNN. α=0 출발이라 하한이 보장된다
run.log("\n" + "=" * 100)
run.log("【P-E】 ★★ P5 — **잔차 부스팅**(관문 아님 · GPU 없으면 건너뛴다)")
run.log("=" * 100)
DLR = {"ran": False, "reason": ""}
try:
    import torch
    import torch.nn as nn
    HAS_CUDA = torch.cuda.is_available()
except Exception as e:
    torch = None; HAS_CUDA = False; DLR["reason"] = f"torch 없음({e})"
if torch is None:
    run.log(f"  ⏭️ 건너뜀 — {DLR['reason']}")
elif not HAS_CUDA and not SMOKE:
    DLR["reason"] = "CUDA 없음"
    run.log(f"  ⏭️ 건너뜀 — {DLR['reason']}. **런타임을 GPU 로 바꾸면 이 절이 돈다**")
else:
    dev = "cuda" if HAS_CUDA else "cpu"
    run.log(f"  ▶ 장치 **{dev}** · {DL_FOLDS}겹 · {DL_EPOCH}에폭 · weight_decay {DL_WD} · "
            f"**pos_weight 없음** · α 초기값 0")
    run.log(f"    ★ Q4-K 실패 원인 셋을 고쳤다 — ① 미수렴(4→{DL_EPOCH}에폭) ② 무정규화"
            f"(→wd {DL_WD}) ③ pos_weight(순위가 아니라 균형 로그손실을 최적화) 제거")
    ordr = sorted(REC_OK, key=lambda r: (BURD[r], r))
    FOLD = {r: i % DL_FOLDS for i, r in enumerate(ordr)}
    # ★ CNN 입력에 **조기성**을 넣는다 — 직전·현재·직후 박동 + 상대 RR 채널
    prev_i = {}; next_i = {}
    for r in RS:
        ii = IDX_ALL[int(r)]
        prev_i[int(r)] = np.r_[ii[0], ii[:-1]]; next_i[int(r)] = np.r_[ii[1:], ii[-1]]
    PREV = np.zeros(len(K), int); NEXT = np.zeros(len(K), int)
    for r in RS:
        ii = IDX_ALL[int(r)]; PREV[ii] = prev_i[int(r)]; NEXT[ii] = next_i[int(r)]
    RELP = np.clip(REL, 0.3, 2.0); RELN = np.clip(post / (BASE12 + 1e-9), 0.3, 2.0)
    def make_x(bi):
        w = np.concatenate([BEAT[PREV[bi]], BEAT[bi], BEAT[NEXT[bi]]], axis=1)
        rr = np.stack([np.repeat(RELP[bi][:, None], LW, 1),
                       np.repeat(RELN[bi][:, None], LW, 1)], axis=1)
        return np.concatenate([w, rr], axis=1).astype("float32")
    NCH = 3 * NL + 2
    class Boost(nn.Module):
        def __init__(self):
            super().__init__()
            self.c = nn.Sequential(
                nn.Conv1d(NCH, 24, 7, 2, 3), nn.GroupNorm(4, 24), nn.ReLU(),
                nn.Conv1d(24, 32, 5, 2, 2), nn.GroupNorm(4, 32), nn.ReLU(),
                nn.Conv1d(32, 32, 3, 2, 1), nn.GroupNorm(4, 32), nn.ReLU(),
                nn.AdaptiveAvgPool1d(1))
            self.e = nn.Linear(32, DL_EMB); self.h = nn.Linear(DL_EMB, 1)
            self.alpha = nn.Parameter(torch.zeros(1))          # ★ α=0 출발
            nn.init.zeros_(self.h.weight); nn.init.zeros_(self.h.bias)
        def forward(self, x, off):
            z = torch.relu(self.e(self.c(x).squeeze(-1)))
            return off + self.alpha * self.h(z).squeeze(-1)
    def cpu_fold(X):
        sc = np.full(len(K), np.nan)
        for f in range(DL_FOLDS):
            te_r = [r for r in REC_OK if FOLD[r] == f]; tr_r = [r for r in REC_OK if FOLD[r] != f]
            tr = np.concatenate([IDXS[r] for r in tr_r]); te = np.concatenate([IDXS[r] for r in te_r])
            mu, sd = X[tr].mean(0), X[tr].std(0) + 1e-9
            lr = LogisticRegression(max_iter=3000, C=1.0).fit((X[tr] - mu) / sd,
                                                              TT_[tr].astype(int))
            sc[tr] = lr.decision_function((X[tr] - mu) / sd)   # 학습 offset 도 필요하다
            sc[te] = lr.decision_function((X[te] - mu) / sd)
        return sc
    OFF = cpu_fold(FEAT["comb"])
    sc_dl = np.full(len(K), np.nan); ALPHA = []
    for f in range(DL_FOLDS):
        te_r = [r for r in REC_OK if FOLD[r] == f]; tr_r = [r for r in REC_OK if FOLD[r] != f]
        tr = np.concatenate([IDXS[r] for r in tr_r]); te = np.concatenate([IDXS[r] for r in te_r])
        wmu = float(BEAT[tr].mean()); wsd = float(BEAT[tr].std()) + 1e-9
        net = Boost().to(dev)
        opt = torch.optim.Adam(net.parameters(), 1e-3, weight_decay=DL_WD)
        lossf = nn.BCEWithLogitsLoss()                          # ★ pos_weight 없음
        for ep in range(DL_EPOCH):
            net.train(); perm = np.random.RandomState(SEED0 + f * 10 + ep).permutation(len(tr))
            for b0 in range(0, len(tr), DL_BATCH):
                bi = tr[perm[b0:b0 + DL_BATCH]]
                x = torch.tensor((make_x(bi) - wmu) / wsd, device=dev)
                o = torch.tensor(OFF[bi].astype("float32"), device=dev)
                y = torch.tensor(TT_[bi].astype("float32"), device=dev)
                opt.zero_grad(); lossf(net(x, o), y).backward(); opt.step()
        net.eval()
        with torch.no_grad():
            for b0 in range(0, len(te), 2048):
                bi = te[b0:b0 + 2048]
                x = torch.tensor((make_x(bi) - wmu) / wsd, device=dev)
                o = torch.tensor(OFF[bi].astype("float32"), device=dev)
                sc_dl[bi] = net(x, o).cpu().numpy()
        ALPHA.append(float(net.alpha.detach().cpu().numpy()[0]))
        run.log(f"    ({time.time()-T0:>5.0f}초) 부스팅 겹 {f+1}/{DL_FOLDS} · α {ALPHA[-1]:+.4f}")
    res = {}
    for nm_, sc in (("cpu_comb(5겹)", OFF), ("boost(5겹)", sc_dl)):
        res[nm_] = dict(ksw=float(np.mean([np.mean([ach_at(sc, r, k) for k in K_SWEEP])
                                           for r in REC_OK])),
                        auc=float(np.mean(list(per_auc(sc).values()))))
    db = boot_pair([np.mean([ach_at(OFF, r, k) for k in K_SWEEP]) for r in REC_OK],
                   [np.mean([ach_at(sc_dl, r, k) for k in K_SWEEP]) for r in REC_OK],
                   SEED0 + 301, NB_BOOT)
    run.log(f"\n  {'팔(5겹 · 같은 자)':<18}{'k-스윕':>10}{'AUROC':>10}")
    for nm_, v in res.items():
        run.log(f"  {nm_:<18}{v['ksw']:>10.4f}{v['auc']:>10.4f}")
    run.log(f"  ▸ `boost − cpu_comb` k-스윕 Δ **{db[0]:+.4f}** [{db[1]:+.4f}, {db[2]:+.4f}] · "
            f"학습된 α 중앙 {np.median(ALPHA):+.4f}")
    run.log(f"  ▸ **α=0 출발이라 하한이 보장된다** — Δ가 음수면 그건 **과적합**이지 구조 문제가 아니다")
    run.log(f"  (Q4-K 하이브리드는 −0.0523 이었다 · 절차 오류 3중이 원인이었다)")
    DLR.update(ran=True, res=res, gain=list(db[:3]), alpha=[float(a) for a in ALPHA])
g_("P5", "(관문 아님)",
   (f"boost − cpu_comb k-스윕 {DLR['gain'][0]:+.4f} · α 중앙 "
    f"{np.median(DLR['alpha']):+.4f}") if DLR["ran"] else f"건너뜀 ({DLR['reason']})")

VSET, NSET = ("V", "E", "F"), ("N", "L", "R", "e", "j", "n")
FPC = {}
for a in ("base", "morph", "pont", "clus", "comb"):
    c = Counter()
    for r in REC_OK:
        idx = IDXS[r]; _, fl = tp_at(L[a], r, MAIN_K)
        for s_ in SYM[idx][fl & (~TT_[idx])]: c[str(s_)] += 1
    FPC[a] = dict(v=sum(c[s] for s in VSET), n=sum(c[s] for s in NSET), tot=sum(c.values()))
run.log(f"\n  ▸ **위양성 구성@300**  {'팔':<8}{'심실기원':>10}{'상심실정상':>12}{'전체':>9}")
for a, d in FPC.items():
    run.log(f"                       {a:<8}{d['v']:>10}{d['n']:>12}{d['tot']:>9}")
run.log(f"    (Q4-L 앵커 V — base 4380 · morph 1444)")
run.log(f"\n  ▸ **운영점·적응증** — 불규칙성(AF 대리) 하위 절반만 배포한다면")
IRR = {r: float(np.sqrt(np.mean(np.diff(pre[IDXS[r]]) ** 2)) / (np.median(pre[IDXS[r]]) + 1e-9))
       for r in REC_OK}
CUT = float(np.median(list(IRR.values())))
LOH = [r for r in REC_OK if IRR[r] < CUT]
run.log(f"    {'팔':<8}" + "".join(f"{'k=' + str(k):>16}" for k in K_SWEEP))
for a in ("base", "morph", "comb"):
    run.log(f"    {a:<8}" + "".join(
        f"{np.mean([ach_at(L[a], r, k) for r in LOH]):>16.4f}" for k in K_SWEEP))
run.log(f"    (규칙 리듬 {len(LOH)}명 · 달성률 = TP/min(S,k) 이라 k ≤ S 면 **정밀도@k**)")
CONFIG["P4"] = P4; CONFIG["P5"] = DLR; CONFIG["fp"] = FPC
CONFIG["indication"] = {a: {str(k): float(np.mean([ach_at(L[a], r, k) for r in LOH]))
                            for k in K_SWEEP} for a in ARMS}
run.save_json("config", CONFIG)


In [ ]:
# CELL 6 — 【P-E】 필요표본 · P6 검산표
run.log("\n" + "=" * 100)
run.log("【P-E】 필요표본 · P6 검산표")
run.log("=" * 100)
eff = om["mean"] - TAB[MAIN_CT][PRIMARY]["thr_deploy"]
n5 = need_super(NRE, om["mde"], eff); n8 = need_super(NRE, om["mde"], eff, True)
bad = (not np.isfinite(eff)) or abs(eff) < om["mde"]
run.log(f"  P2 주 관문(k-스윕) 효과-문턱 {eff:+.4f} · 반폭 {om['mde']:.4f} · n(50%) {n5:.0f} · "
        f"n(80%) {n8:.0f}  "
        + ("★ **해석 불가**(R41 ②)" if bad else
           ("이미 충분하다" if n8 <= NRE else "표본이 더 필요하다")))

CHECK = [
    dict(claim="★★★ Q4-K 의 딥러닝 패배는 **딥러닝 탓이 아니었다** — 절차 오류 3중이다",
         num="같은 특징·같은 데이터로 절차만 바꾸면 sklearn lbfgs 0.8434 vs "
             "Adam 4에폭·무정규화·pos_weight 6.1 **0.7008** — **−0.14 가 순전히 최적화 절차**다"
             "(40에폭+wd 로 0.8433 회복). 게다가 CNN 입력이 R 정렬 단일 박동이라 **RR 이 "
             "입력에 없었다**(`dl_wave` 0.4940 이 그 결과). 유효 표본도 레코드 56 이다",
         assume="본 런은 **α=0 출발 잔차 부스팅**으로 하한을 보장한다",
         iffalse="★★★ 그래서 이번 Δ가 음수면 그건 **과적합**이지 「딥러닝이 안 통한다」가 아니다"),
    dict(claim=f"★★★ P2 주 관문 — P-on-T 의 k-스윕 값 {om['mean']:+.4f} → {VERD['P2']}",
         num=f"두 팔 모두 {FEAT['pont'].shape[1]}차원 · **형태 위에** 얹었다 · 영점 "
             f"{NSTAT[MAIN_CT][PRIMARY][0]:+.4f}"
             f"(rep {NREP[MAIN_CT][PRIMARY]['mean']:+.4f}) · 문턱 {td:+.4f} · "
             f"AUROC 로는 {oa['mean']:+.4f}",
         assume="델리네이션·템플릿·잔차화는 **라벨을 안 쓴다**(R22)",
         iffalse="★★ P-on-T 가 ❌ 면 형태 축이 **포화**라는 뜻이고, 남는 건 **적응증·운영점**이다"),
    dict(claim="★★★ 관문 자체가 **통과 불가**에 가까웠다 — 주 지표를 k-스윕으로 바꿨다",
         num=f"Q4-K: morph 달성률 {REF['q4k']['morph']['ach']} → 남은 여지 0.0982 인데 문턱 "
             f"**+{REF['q4k']['thr']:.4f}**(53%). 본 런 k-스윕 문턱 **{td:+.4f}** · "
             f"항등식 달성률 = TP/min(S,k) 이라 k <= S 면 정밀도@k",
         assume=f"k-스윕 {list(K_SWEEP)} 을 **데이터 보기 전** 고정(R34 ②)",
         iffalse="★★ 네 점 평균이라 **영점 분산도 준다** — 단일 k 상위통계보다 안정적이다"),
    dict(claim=f"★★ **중복 감사 v2** — 레코드 내 순위로 재니 중복 {n_dup}/{len(DUP)}",
         num=f"회고 검증: Q4-K 의 `tp_over_rr` 는 풀링 R² {REF['q4k']['tp_r2']}(=새 축)인데 "
             f"단변량 {REF['q4k']['tp_uni']}(RR 최고 {REF['q4k']['base_uni']} 급)였다 — "
             f"**둘이 같이 큰 게 단조 중복의 지문**. 본 런 새 자로 레코드내|ρ| "
             f"{CONFIG['P1']['retro']['rho']:.4f}",
         assume=f"문턱 레코드내 |ρ| > {DUP_RHO} 를 사전 고정",
         iffalse="★★ 매크로 채점은 **레코드 내 순위**만 쓴다 — 풀링 지표는 레코드 간 변동에 "
                 "지배돼 중복을 놓친다"),
    dict(claim="★★ 형태는 **유일하게 확립된 축**이고, 무엇을 보는지도 안다",
         num=f"`morph − base` k-스윕 {OBS['morph-base'][PRIMARY]['mean']:+.4f}(Q4-L +0.1570) · "
             f"달성률@300 "
             f"{np.mean(list(ACH['morph'].values())) - np.mean(list(ACH['base'].values())):+.4f}"
             f"(Q4-J·Q4-K +0.0944) · 위양성 심실기원 {FPC['base']['v']} → {FPC['morph']['v']} · "
             f"단변량 최고 `p_energy_ratio`(선행 P 부재) > `corr_st_min`(2차성 ST-T) · "
             f"k=50 이득이 k=300 의 2.3배 — **순위 맨 위에서 V 를 밀어낸다**. 그 위에 얹은 "
             f"P-on-T {OBS['pont-morph'][PRIMARY]['mean']:+.4f} · 군집 "
             f"{OBS['clus-morph'][PRIMARY]['mean']:+.4f}",
         assume="같은 코호트·같은 LORO — 네 런의 수치가 직접 비교 가능하다",
         iffalse="★ 둘 다 안 얹히면 **박동 단위 특징은 포화**이고, 남는 건 **적응증·운영점**이다"),
    dict(claim=("★★ 잔차 부스팅 — " + (f"boost − cpu_comb {DLR['gain'][0]:+.4f}"
                                        if DLR["ran"] else f"건너뜀({DLR['reason']})")),
         num=(f"α 중앙 {np.median(DLR['alpha']):+.4f} · " +
              " · ".join(f"{k} {v['ksw']:.4f}" for k, v in DLR["res"].items())
              if DLR["ran"] else "GPU 런타임에서 다시 돌리면 된다"),
         assume="**α=0 출발**이라 초기 상태가 정확히 `cpu_comb` 다 — 하한이 보장된다",
         iffalse="★★ 그래서 Δ<0 이면 **과적합**이고, α 가 0 근처면 **CNN 이 더할 게 없다**는 뜻이다"),
    dict(claim="★ 새 축 둘의 임상 근거",
         num=f"**P-on-T** — 매우 이른 PAC 은 P′ 가 선행 T 에 겹쳐 **직전 박동의 T 가 변형**된다"
             f"(단변량 최고 {max(DUP[n]['uni'] for n in PONAMES):.4f}). "
             f"**군집** — 한 환자의 S 는 소수 초점이라 서로 닮았다"
             f"(최고 {max(DUP[n]['uni'] for n in CLNAMES):.4f} · **무라벨**)",
         assume="군집은 KMeans k=3 · 레코드별 · **라벨을 안 쓴다**(R22)",
         iffalse="★ 둘 다 형태 위에 안 얹히면 **박동 단위 특징은 포화**라는 결론이 된다"),
]
for i, ck in enumerate(CHECK, 1):
    run.log(f"\n  [{i}] **{ck['claim']}**")
    run.log(f"      근거   {ck['num']}")
    run.log(f"      가정   {ck['assume']}")
    run.log(f"      틀리면 {ck['iffalse']}")
CONFIG["need"] = dict(effect=float(eff), half=float(om["mde"]), sup50=float(n5),
                      sup80=float(n8), uninterpretable=bool(bad))
CONFIG["P6"] = CHECK
run.save_json("config", CONFIG)


In [ ]:
# CELL 7 — 【P-F】 그림 · 요약 · 마무리
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import Image, display
fig, ax = plt.subplots(1, 3, figsize=(16.5, 4.6))
xs = np.arange(len(ARMS))
ax[0].bar(xs - 0.2, [np.mean(list(KSW[a].values())) for a in ARMS], 0.4, label="k-sweep")
ax[0].bar(xs + 0.2, [np.mean(list(ACH[a].values())) for a in ARMS], 0.4, label="achievement@300")
ax[0].set_xticks(xs); ax[0].set_xticklabels(ARMS, fontsize=7, rotation=20)
ax[0].legend(fontsize=7); ax[0].grid(alpha=.3, axis="y")
ax[0].set_title("primary (k-sweep) vs achievement@300", fontsize=9)

nm = [c[0] for c in CONTRASTS]
vv = [OBS[n][PRIMARY]["mean"] for n in nm]
lo = [vv[i] - OBS[n][PRIMARY]["lo"] for i, n in enumerate(nm)]
hi = [OBS[n][PRIMARY]["hi"] - vv[i] for i, n in enumerate(nm)]
ax[1].errorbar(vv, np.arange(len(nm)), xerr=[lo, hi], fmt="o", capsize=5, color="tab:blue")
ax[1].scatter([NSTAT[n][PRIMARY][0] for n in nm], np.arange(len(nm)), marker="x", s=45,
              color="tab:gray", label="null (raw)")
ax[1].axvline(0, color="k", lw=.9)
ax[1].set_yticks(range(len(nm))); ax[1].set_yticklabels(nm, fontsize=7)
ax[1].set_xlabel("k-sweep achievement delta"); ax[1].legend(fontsize=7)
ax[1].set_title("P2/P3/P5 : contrasts", fontsize=9); ax[1].grid(alpha=.3, axis="x")

bn = list(BLOCKS.keys())
ax[2].barh(np.arange(len(bn)), [P4[b]["ksw"][0] for b in bn], color="tab:green")
ax[2].set_yticks(range(len(bn)))
ax[2].set_yticklabels(bn, fontsize=8)
ax[2].axvline(0, color="k", lw=.9); ax[2].set_xlabel("k-sweep delta vs morph")
ax[2].set_title("P4 : new axes on top of morphology", fontsize=9); ax[2].grid(alpha=.3, axis="x")
fig.tight_layout()
PNG = run.save_fig("q4m_pont_cluster_boost", fig)
plt.close(fig); display(Image(PNG))

run.log("\n" + "=" * 100)
run.log("요약")
run.log("=" * 100)
ok_ = lambda k: VERD.get(k, "").startswith("✅")
for g in READ_ORDER[:6]:
    run.log(f"  {g:<5}{VERD.get(g, '(관문 아님)')}")
run.log("")
run.log(f"  ★★★ **자 검증** — QRS {QRS_MS:.1f}ms(Q4-K {REF['q4k']['qrs_ms']}) · "
        f"PR {PR_MS:.1f}ms({REF['q4k']['pr_ms']}) · **역위 P {inv_rate:.1%}** · "
        f"P|corr| N {pn:.3f} > V {pv:.3f}")
run.log(f"  ★★★ **P2 주 관문** — `{MAIN_CT}` k-스윕 {om['mean']:+.4f} "
        f"[{om['lo']:+.4f}, {om['hi']:+.4f}] → {VERD['P2']}")
run.log(f"  ★★ **P3** — `comb − morph` {OBS['comb-morph'][PRIMARY]['mean']:+.4f} → {VERD['P3']} · "
        f"`clus − cshuf` {OBS['clus-cshuf'][PRIMARY]['mean']:+.4f}")
run.log(f"  ★★ **P5 잔차 부스팅** — " + (f"boost − cpu_comb {DLR['gain'][0]:+.4f} · α 중앙 "
        f"{np.median(DLR['alpha']):+.4f}" if DLR["ran"] else f"건너뜀({DLR['reason']})"))
run.log(f"  ★★ **P4 블록** — " + " · ".join(f"{n} {P4[n]['ksw'][0]:+.4f}" for n in BLOCKS))
run.log(f"  ★★ **P1 중복 감사 v2** — 중복 {n_dup}/{len(DUP)} · 회고 `tp_over_rr` "
        f"레코드내|ρ| {CONFIG['P1']['retro']['rho']:.4f}")
run.log(f"  ▸ k-스윕 — " + " · ".join(f"{a} {np.mean(list(KSW[a].values())):.4f}" for a in ARMS))
run.log(f"  ▸ 위양성 심실기원 — " + " · ".join(f"{a} {FPC[a]['v']}" for a in FPC))
run.log(f"  ▸ **적응증(규칙 절반)** — " + " · ".join(
    f"k={k} base {CONFIG['indication']['base'][str(k)]:.3f} → comb "
    f"{CONFIG['indication']['comb'][str(k)]:.3f}" for k in K_SWEEP))

run.finish({
    "exp_id": "quest46_q4m_pont_cluster_boost",
    "metric": "ksweep_pont_minus_pshuf",
    "value": float(om["mean"]),
    "passed": bool(ok_("P0") and ok_("P2")),
    "summary": ("형태가 **무엇을 보는지**가 정량화됐다 — 단변량 최고가 `p_energy_ratio`"
                "(V 는 방실해리라 선행 P 가 없다) > `corr_st_min`(2차성 ST-T)이고 `qrs_width` 가 "
                "최하위이며, 이득이 k=50 에서 +0.2230 · k=300 에서 +0.0950 으로 **k 가 작을수록 "
                "크다**. 즉 형태는 **S 를 더 찾는 게 아니라 순위 맨 위에서 V 를 밀어낸다**"
                "(심실기원 위양성 4380 → 1444). 그리고 Q4-K 의 딥러닝 패배는 **딥러닝 탓이 "
                "아니었다** — 같은 특징으로 절차만 바꾸면 sklearn lbfgs 0.8434 vs Adam 4에폭·"
                "무정규화·pos_weight **0.7008** 로 **−0.14 가 순전히 최적화 절차**였고(40에폭+wd 로 "
                "0.8433 회복), CNN 입력이 R 정렬 단일 박동이라 **RR 이 입력에 아예 없었다**. "
                "이번엔 **α=0 출발 잔차 부스팅**(`logit = logit_cpu(고정) + α·CNN(다중박동+RR채널)`)"
                "으로 하한을 보장한다. 새 축 둘 — **P-on-T**(매우 이른 PAC 은 P′ 가 선행 T 에 겹쳐 "
                "**직전 박동의 T 가 변형**된다. 지금까지 모든 특징은 그 박동 **자신의 창**만 봤다)와 "
                "**환자 내 무라벨 군집**(한 환자의 S 는 소수 초점이라 서로 닮았다)을 **형태 위에** "
                "얹어 차원 동일 대조로 판정한다."),
    "verdicts": VERD, "notes": NOTE, "rule_check": RULE_CHECK,
    "cohort": CONFIG.get("cohort", {}), "P0": CONFIG.get("P0", {}),
    "P1": CONFIG.get("P1", {}), "P2": CONFIG.get("P2", {}), "P4": CONFIG.get("P4", {}),
    "P5": CONFIG.get("P5", {}), "indication": CONFIG.get("indication", {}),
    "fp": CONFIG.get("fp", {}), "need": CONFIG.get("need", {}),
    "P6": CONFIG.get("P6", []), "fig": PNG})
run.log(f"\n저장 완료 — {run.dir}")
run.log("다음: `python pipelines/ingest_run.py --results result.json "
        "--notebook notebooks/quest46_q4m_pont_cluster_boost.ipynb`")
